# Realistic Human Face Generation using GAN (Lightweight DCGAN / LSGAN)

**Objective:** build a Generative Adversarial Network that generates realistic synthetic human faces from random noise.

**What this notebook does (end to end):**
1. Loads a human-face dataset (CelebA subset) and preprocesses it (crop, resize, normalize).
2. Builds a lightweight Generator and Discriminator (64x64 DCGAN architecture).
3. Trains two GAN variants with adversarial learning: **DCGAN** (BCE loss) and **LSGAN** (least-squares loss).
4. Monitors losses, saves generated samples every epoch, and scores quality with a cheap metric.
5. Saves the **best checkpoint of each model** plus the **best overall model**.
6. Compares results over time and presents final generated faces.
7. Deploys a **Gradio** app on a chosen port where the user can pick the model.

**Kaggle setup (keeps cost low):**
- Accelerator: **GPU T4 x1** (one GPU is enough; total training takes only a few minutes).
- Add Input: search for the dataset **"CelebA Dataset"** (by jessicali9530) and add it. No downloads are needed during the run.
- Internet: only needed at the very end for the public Gradio link (and `pip install gradio` if it is missing).

## 1. Imports, reproducibility and device

We import the libraries, define a seed helper so results are reproducible, and select the GPU if it is available.

In [ ]:
import os, glob, json, time, shutil, random
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import torch
import torch.nn as nn
import torchvision.utils as vutils
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Configuration

All hyperparameters live in one dictionary so they are easy to change.

- `img_size=64` and `num_images=20000` keep memory, time and GPU usage small.
- `epochs=20` is enough to get recognizable faces with DCGAN on a T4 in a few minutes.
- `eval_every` controls how often the quality metric is computed (this is also when the best checkpoint is saved).

In [ ]:
CFG = dict(
    img_size=64,
    latent_dim=100,
    gen_features=64,
    disc_features=64,
    num_images=20000,
    batch_size=128,
    epochs=20,
    lr=2e-4,
    beta1=0.5,
    eval_every=2,
    eval_samples=3000,
    seed=42,
)

OUT_DIR = "/kaggle/working"
SAMPLES_DIR = f"{OUT_DIR}/samples"
MODELS_DIR = f"{OUT_DIR}/models"
os.makedirs(SAMPLES_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

set_seed(CFG["seed"])
# Fixed noise: the same 64 latent vectors are used every epoch, so progress is visually comparable.
FIXED_Z = torch.randn(64, CFG["latent_dim"], 1, 1, generator=torch.Generator().manual_seed(CFG["seed"])).to(device)

## 3. Locate the dataset

The notebook searches `/kaggle/input` for the first folder that contains many images, so it works regardless of the exact dataset folder layout.

In [ ]:
def find_image_folder(root="/kaggle/input"):
    for dirpath, _, files in os.walk(root):
        imgs = sorted(f for f in files if f.lower().endswith((".jpg", ".jpeg", ".png")))
        if len(imgs) > 100:
            return dirpath, imgs
    raise FileNotFoundError("No face images found. Add the 'CelebA Dataset' to this notebook via Add Input.")


IMG_DIR, ALL_FILES = find_image_folder()
print("Image folder:", IMG_DIR)
print("Total images available:", len(ALL_FILES))

## 4. Load and preprocess the faces

Preprocessing steps (as required by the project):
- **Crop:** a centered square crop removes most of the background and keeps the face.
- **Resize:** the crop is resized to 64x64.
- **Normalize:** pixels are scaled to the range [-1, 1] later on the GPU (matching the Generator's `tanh` output).

A random subset of `num_images` is loaded once with threads and kept in memory as a compact `uint8` tensor (about 245 MB), which makes every epoch fast.

In [ ]:
def load_and_preprocess(path, size=CFG["img_size"]):
    img = Image.open(path).convert("RGB")
    w, h = img.size
    side = int(min(w, h) * 0.82)                      # centered square crop around the face
    left, top = (w - side) // 2, (h - side) // 2
    img = img.crop((left, top, left + side, top + side)).resize((size, size), Image.LANCZOS)
    return np.asarray(img)


rng = np.random.default_rng(CFG["seed"])
chosen = rng.choice(len(ALL_FILES), size=min(CFG["num_images"], len(ALL_FILES)), replace=False)
paths = [os.path.join(IMG_DIR, ALL_FILES[i]) for i in chosen]

t0 = time.time()
with ThreadPoolExecutor(max_workers=4) as pool:
    arrays = list(tqdm(pool.map(load_and_preprocess, paths), total=len(paths), desc="Loading faces"))

DATA = torch.from_numpy(np.stack(arrays)).permute(0, 3, 1, 2).contiguous()   # (N, 3, 64, 64) uint8
del arrays
print(f"Dataset tensor: {tuple(DATA.shape)}, dtype={DATA.dtype}, loaded in {time.time() - t0:.1f}s")

## 5. Batch generator and display helpers

- `batches` shuffles the data each epoch, moves a batch to the GPU, **normalizes to [-1, 1]**, and applies a random horizontal flip as light augmentation.
- `denorm` converts images from [-1, 1] back to [0, 1] for display and saving.
- `show_grid` plots a grid of images.

In [ ]:
def batches(data, batch_size):
    idx = torch.randperm(len(data))
    for i in range(0, len(data) - batch_size + 1, batch_size):
        x = data[idx[i:i + batch_size]].to(device, non_blocking=True).float().div(127.5).sub(1.0)
        flip = torch.rand(x.size(0), device=device) < 0.5
        yield torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)


def denorm(x):
    return (x * 0.5 + 0.5).clamp(0, 1)


def show_grid(x, title="", nrow=8, figsize=(8, 8)):
    grid = vutils.make_grid(denorm(x.detach().cpu()), nrow=nrow, padding=2)
    plt.figure(figsize=figsize)
    plt.imshow(grid.permute(1, 2, 0).numpy())
    plt.axis("off")
    plt.title(title)
    plt.show()

## 6. Look at real training faces

A quick sanity check that the cropping and resizing worked and that the data looks like faces.

In [ ]:
sample_real = next(batches(DATA, 64))
show_grid(sample_real, title="Real training faces (after preprocessing)")

## 7. Generator, Discriminator and utility functions

**Generator:** takes a 100-dim random vector and upsamples it with transposed convolutions (1x1 -> 4 -> 8 -> 16 -> 32 -> 64) using BatchNorm + ReLU, ending with `tanh`.

**Discriminator:** a small CNN that downsamples the image (64 -> 32 -> 16 -> 8 -> 4 -> 1) using strided convolutions + LeakyReLU, and outputs a single real/fake **logit** (no sigmoid, the loss functions handle that).

Also defined here (and reused later for training, evaluation, display and deployment, so there is no duplicated code):
- `generate_faces`: samples faces from a Generator, optionally with a fixed seed.
- `load_generator`: rebuilds a Generator from a saved checkpoint.

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim=100, features=64):
        super().__init__()
        self.latent_dim, self.features = latent_dim, features
        f = features
        self.net = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, f * 8, 4, 1, 0, bias=False), nn.BatchNorm2d(f * 8), nn.ReLU(True),   # 4x4
            nn.ConvTranspose2d(f * 8, f * 4, 4, 2, 1, bias=False), nn.BatchNorm2d(f * 4), nn.ReLU(True),        # 8x8
            nn.ConvTranspose2d(f * 4, f * 2, 4, 2, 1, bias=False), nn.BatchNorm2d(f * 2), nn.ReLU(True),        # 16x16
            nn.ConvTranspose2d(f * 2, f, 4, 2, 1, bias=False), nn.BatchNorm2d(f), nn.ReLU(True),                # 32x32
            nn.ConvTranspose2d(f, 3, 4, 2, 1, bias=False), nn.Tanh(),                                           # 64x64
        )

    def forward(self, z):
        return self.net(z)


class Discriminator(nn.Module):
    def __init__(self, features=64):
        super().__init__()
        f = features
        self.net = nn.Sequential(
            nn.Conv2d(3, f, 4, 2, 1, bias=False), nn.LeakyReLU(0.2, True),                                       # 32x32
            nn.Conv2d(f, f * 2, 4, 2, 1, bias=False), nn.BatchNorm2d(f * 2), nn.LeakyReLU(0.2, True),           # 16x16
            nn.Conv2d(f * 2, f * 4, 4, 2, 1, bias=False), nn.BatchNorm2d(f * 4), nn.LeakyReLU(0.2, True),       # 8x8
            nn.Conv2d(f * 4, f * 8, 4, 2, 1, bias=False), nn.BatchNorm2d(f * 8), nn.LeakyReLU(0.2, True),       # 4x4
            nn.Conv2d(f * 8, 1, 4, 1, 0, bias=False),                                                           # 1x1 logit
        )

    def forward(self, x):
        return self.net(x).view(-1)


def init_weights(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(m.weight, 0.0, 0.02)
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.normal_(m.weight, 1.0, 0.02)
        nn.init.zeros_(m.bias)


def build_models():
    G = Generator(CFG["latent_dim"], CFG["gen_features"]).to(device)
    D = Discriminator(CFG["disc_features"]).to(device)
    G.apply(init_weights)
    D.apply(init_weights)
    return G, D


@torch.no_grad()
def generate_faces(G, n, seed=None):
    G.eval()
    gen = torch.Generator()
    gen.manual_seed(int(seed)) if seed is not None else gen.seed()
    z = torch.randn(n, G.latent_dim, 1, 1, generator=gen).to(next(G.parameters()).device)
    return G(z)


def load_generator(path, map_location="cpu"):
    ckpt = torch.load(path, map_location=map_location)
    G = Generator(ckpt["latent_dim"], ckpt["gen_features"])
    G.load_state_dict(ckpt["state_dict"])
    return G.to(map_location).eval(), ckpt


_G, _D = build_models()
count = lambda m: sum(p.numel() for p in m.parameters()) / 1e6
print(f"Generator parameters: {count(_G):.2f}M | Discriminator parameters: {count(_D):.2f}M")
del _G, _D

## 8. Cheap quality metric (Frechet distance on small pixel features)

The official FID needs a pretrained Inception network (an extra download and more GPU time). To stay light, we use a **proxy**: the Frechet distance between real and generated images, computed on 16x16 downsampled pixels (768 features) with plain NumPy.

- Lower is better.
- It is **not** the official FID, so only compare numbers produced by this notebook with each other.
- It is used to pick the **best checkpoint** during training and the **best model** overall.

In [ ]:
def image_features(x):
    return nn.functional.adaptive_avg_pool2d(x, 16).flatten(1).double().cpu().numpy()


def gaussian_stats(features):
    return features.mean(0), np.cov(features, rowvar=False)


def frechet_distance(m1, s1, m2, s2):
    diff = m1 - m2
    eigvals = np.linalg.eigvals(s1 @ s2)
    trace_sqrt = np.sqrt(np.clip(eigvals.real, 0, None)).sum()
    return float(diff @ diff + np.trace(s1) + np.trace(s2) - 2.0 * trace_sqrt)


n_eval = CFG["eval_samples"]
real_batch = DATA[:n_eval].float().div(127.5).sub(1.0)
REAL_STATS = gaussian_stats(image_features(real_batch))
del real_batch


@torch.no_grad()
def evaluate_generator(G):
    G.eval()
    feats = []
    for start in range(0, n_eval, 500):
        z = torch.randn(min(500, n_eval - start), CFG["latent_dim"], 1, 1, device=device)
        feats.append(image_features(G(z)))
    return frechet_distance(*gaussian_stats(np.concatenate(feats)), *REAL_STATS)

## 9. Training function (adversarial learning)

One function trains either GAN variant:

- **DCGAN (`bce`)**: binary cross-entropy on logits with one-sided label smoothing (real label = 0.9) for more stable training.
- **LSGAN (`lsgan`)**: least-squares (MSE) loss, which often gives smoother gradients.

Every iteration: (1) update the Discriminator on real and fake faces, (2) update the Generator so the Discriminator labels its fakes as real.

Every epoch it saves a sample grid from the fixed noise and logs the losses. Every `eval_every` epochs it computes the quality metric and **saves the best Generator checkpoint** (with its config, so it can be reloaded anywhere).

In [ ]:
def train_gan(name, loss_type):
    set_seed(CFG["seed"])
    G, D = build_models()
    opt_g = torch.optim.Adam(G.parameters(), lr=CFG["lr"], betas=(CFG["beta1"], 0.999))
    opt_d = torch.optim.Adam(D.parameters(), lr=CFG["lr"], betas=(CFG["beta1"], 0.999))

    if loss_type == "bce":
        criterion, real_target = nn.BCEWithLogitsLoss(), 0.9
    else:
        criterion, real_target = nn.MSELoss(), 1.0

    sample_dir = f"{SAMPLES_DIR}/{name}"
    os.makedirs(sample_dir, exist_ok=True)
    ckpt_path = f"{MODELS_DIR}/{name}_best.pth"
    hist = dict(g_loss=[], d_loss=[], d_real=[], d_fake=[], metric_epochs=[], metric=[])
    best_metric, best_epoch = float("inf"), 0

    for epoch in range(1, CFG["epochs"] + 1):
        G.train(); D.train()
        t0, n_batches = time.time(), 0
        sums = torch.zeros(4, device=device)
        for real in batches(DATA, CFG["batch_size"]):
            b = real.size(0)
            fake = G(torch.randn(b, CFG["latent_dim"], 1, 1, device=device))

            # 1) Discriminator step
            opt_d.zero_grad(set_to_none=True)
            out_real, out_fake = D(real), D(fake.detach())
            loss_d = criterion(out_real, torch.full_like(out_real, real_target)) + criterion(out_fake, torch.zeros_like(out_fake))
            loss_d.backward()
            opt_d.step()

            # 2) Generator step
            opt_g.zero_grad(set_to_none=True)
            out_gen = D(fake)
            loss_g = criterion(out_gen, torch.ones_like(out_gen))
            loss_g.backward()
            opt_g.step()

            sums += torch.stack([loss_g.detach(), loss_d.detach(), out_real.detach().mean(), out_fake.detach().mean()])
            n_batches += 1

        g_l, d_l, d_r, d_f = (sums / n_batches).tolist()
        for key, val in zip(["g_loss", "d_loss", "d_real", "d_fake"], [g_l, d_l, d_r, d_f]):
            hist[key].append(val)

        # Save generated samples from the fixed noise
        G.eval()
        with torch.no_grad():
            vutils.save_image(denorm(G(FIXED_Z)), f"{sample_dir}/epoch_{epoch:03d}.png", nrow=8, padding=2)

        msg = f"[{name}] epoch {epoch:02d}/{CFG['epochs']} | G {g_l:.3f} | D {d_l:.3f} | {time.time() - t0:.1f}s"
        if epoch % CFG["eval_every"] == 0 or epoch == CFG["epochs"]:
            metric = evaluate_generator(G)
            hist["metric_epochs"].append(epoch)
            hist["metric"].append(metric)
            msg += f" | metric {metric:.4f}"
            if metric < best_metric:
                best_metric, best_epoch = metric, epoch
                torch.save(dict(name=name, loss_type=loss_type, state_dict=G.state_dict(),
                                latent_dim=CFG["latent_dim"], gen_features=CFG["gen_features"],
                                epoch=epoch, metric=metric), ckpt_path)
                msg += "  <- saved best"
        print(msg)

    return dict(history=hist, best_metric=best_metric, best_epoch=best_epoch, ckpt_path=ckpt_path, sample_dir=sample_dir)


RESULTS = {}

## 10. Train model 1: DCGAN (BCE loss)

In [ ]:
RESULTS["DCGAN"] = train_gan("DCGAN", "bce")
print(f"Best DCGAN: epoch {RESULTS['DCGAN']['best_epoch']} | metric {RESULTS['DCGAN']['best_metric']:.4f}")

## 11. Train model 2: LSGAN (least-squares loss)

Same lightweight architecture, different adversarial loss. This gives us two models to compare and to choose from in the deployment interface.

In [ ]:
RESULTS["LSGAN"] = train_gan("LSGAN", "lsgan")
print(f"Best LSGAN: epoch {RESULTS['LSGAN']['best_epoch']} | metric {RESULTS['LSGAN']['best_metric']:.4f}")

## 12. Monitor losses and quality over time

Left to right: Generator loss, Discriminator loss, mean Discriminator output on real vs fake faces, and the quality metric (lower is better).

In a healthy GAN the two losses stay in balance (neither goes to zero), and the metric trends downward.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for row, (name, res) in enumerate(RESULTS.items()):
    h, epochs = res["history"], range(1, CFG["epochs"] + 1)
    axes[row, 0].plot(epochs, h["g_loss"]); axes[row, 0].set_title(f"{name} - Generator loss")
    axes[row, 1].plot(epochs, h["d_loss"], color="tab:orange"); axes[row, 1].set_title(f"{name} - Discriminator loss")
    axes[row, 2].plot(epochs, h["d_real"], label="D(real)"); axes[row, 2].plot(epochs, h["d_fake"], label="D(fake)")
    axes[row, 2].set_title(f"{name} - Discriminator outputs"); axes[row, 2].legend()
    axes[row, 3].plot(h["metric_epochs"], h["metric"], marker="o", color="tab:green"); axes[row, 3].set_title(f"{name} - Quality metric")
    for ax in axes[row]:
        ax.set_xlabel("Epoch"); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/training_curves.png", dpi=120)
plt.show()

## 13. Compare generated faces over time

Each row is one model, each column an epoch. All images come from the **same fixed noise**, so you can see how the same latent vectors turn from blobs into faces during training.

In [ ]:
E = CFG["epochs"]
epochs_to_show = sorted({1, max(1, E // 4), max(1, E // 2), max(1, 3 * E // 4), E})

fig, axes = plt.subplots(len(RESULTS), len(epochs_to_show), figsize=(4 * len(epochs_to_show), 4.2 * len(RESULTS)))
for r, (name, res) in enumerate(RESULTS.items()):
    for c, ep in enumerate(epochs_to_show):
        axes[r, c].imshow(Image.open(f"{res['sample_dir']}/epoch_{ep:03d}.png"))
        axes[r, c].set_title(f"{name} - epoch {ep}")
        axes[r, c].axis("off")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/progress_comparison.png", dpi=120)
plt.show()

## 14. Select and save the best model

We compare the best checkpoint of each model using the quality metric and copy the winner to `best_overall.pth`.

A small `registry.json` file records every saved model (path, epoch, metric). The Gradio app reads this file to build its model dropdown.

In [ ]:
summary = {name: dict(path=res["ckpt_path"], epoch=res["best_epoch"], metric=round(res["best_metric"], 4)) for name, res in RESULTS.items()}
best_name = min(summary, key=lambda n: summary[n]["metric"])

best_path = f"{MODELS_DIR}/best_overall.pth"
shutil.copy(summary[best_name]["path"], best_path)

registry = dict(models=summary, best_overall=dict(name=best_name, path=best_path))
with open(f"{MODELS_DIR}/registry.json", "w") as f:
    json.dump(registry, f, indent=2)

for name, info in summary.items():
    print(f"{name:6s} | best epoch {info['epoch']:2d} | metric {info['metric']:.4f}")
print(f"\nBest overall model: {best_name}  ->  {best_path}")

## 15. Final generated faces

We load the best overall model from disk (exactly like the deployment app will) and present 64 newly generated faces from random noise.

In [ ]:
G_best, ckpt = load_generator(best_path, map_location=device)
faces = generate_faces(G_best, 64, seed=123)
show_grid(faces, title=f"Final generated faces - {ckpt['name']} (epoch {ckpt['epoch']})", figsize=(9, 9))
vutils.save_image(denorm(faces), f"{OUT_DIR}/final_faces.png", nrow=8, padding=2)

## 16. Deployment with Gradio (choose the model from the interface)

The app reads `registry.json` and offers a dropdown with **Best overall** plus every saved model. The user can also choose how many faces to generate and a seed (or tick "random" for new faces every time).

- Inference runs on the **CPU**, so the app does not use any GPU.
- Models are loaded once and cached.
- `gr.close_all()` closes any older Gradio app that still holds the port, and `debug=True` keeps the cell connected so the public link stays alive while the cell is running (press Stop when you are done).
- The app listens on `server_name="0.0.0.0"` and the port set in `PORT`.
- On Kaggle, `localhost` is not reachable from your browser, so `SHARE = True` creates a temporary public link (this needs Internet ON). Set `SHARE = False` if you run the notebook on your own machine and open `http://localhost:PORT`.

In [ ]:
try:
    import gradio as gr
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gradio"])
    import gradio as gr

PORT = 7860
SHARE = True

with open(f"{MODELS_DIR}/registry.json") as f:
    REGISTRY = json.load(f)

BEST_LABEL = f"Best overall ({REGISTRY['best_overall']['name']})"
MODEL_PATHS = {BEST_LABEL: REGISTRY["best_overall"]["path"]}
MODEL_PATHS.update({name: info["path"] for name, info in REGISTRY["models"].items()})
GENERATOR_CACHE = {}


def get_generator(choice):
    if choice not in GENERATOR_CACHE:
        GENERATOR_CACHE[choice] = load_generator(MODEL_PATHS[choice], map_location="cpu")
    return GENERATOR_CACHE[choice]


def generate_for_ui(model_choice, n_images, seed, randomize):
    G, ckpt = get_generator(model_choice)
    faces = generate_faces(G, int(n_images), seed=None if randomize else int(seed))
    images = [
        Image.fromarray((denorm(f).permute(1, 2, 0).numpy() * 255).astype(np.uint8)).resize((256, 256), Image.BICUBIC)
        for f in faces
    ]
    info = f"Model: {ckpt['name']} | best epoch: {ckpt['epoch']} | quality metric (lower is better): {ckpt['metric']:.4f}"
    return images, info


with gr.Blocks(title="GAN Face Generator") as demo:
    gr.Markdown("# Realistic Human Face Generation using GAN\nChoose a model, then generate synthetic faces from random noise.")
    with gr.Row():
        with gr.Column(scale=1):
            model_dd = gr.Dropdown(choices=list(MODEL_PATHS), value=BEST_LABEL, label="Model")
            n_slider = gr.Slider(1, 16, value=8, step=1, label="Number of faces")
            seed_box = gr.Number(value=42, precision=0, label="Seed")
            random_cb = gr.Checkbox(value=True, label="Random seed (new faces every click)")
            run_btn = gr.Button("Generate faces", variant="primary")
            info_box = gr.Textbox(label="Model info", interactive=False)
        with gr.Column(scale=2):
            gallery = gr.Gallery(label="Generated faces", columns=4, height="auto")
    run_btn.click(generate_for_ui, inputs=[model_dd, n_slider, seed_box, random_cb], outputs=[gallery, info_box])

gr.close_all()   # free the port if an older app is still running
demo.launch(server_name="0.0.0.0", server_port=PORT, share=SHARE, debug=True)

## 17. Output files and summary

Files saved in `/kaggle/working`:
- `models/DCGAN_best.pth`, `models/LSGAN_best.pth`: best checkpoint of each model.
- `models/best_overall.pth` and `models/registry.json`: the overall winner and the model list used by the app.
- `samples/<model>/epoch_XXX.png`: generated samples saved during training.
- `training_curves.png`, `progress_comparison.png`, `final_faces.png`: results figures.

**Why this setup is lightweight:** 64x64 images, a 20k-image subset kept in memory, small DCGAN networks (a few million parameters), only 20 epochs, no pretrained downloads, and CPU-only inference in the app.

**Possible improvements:** train for more epochs or on more images, add an EMA copy of the Generator, or compute the official FID if extra time and internet are available.